**MGMT298D: Science and Strategy of AI**

# Week 6B: Building a Tiny LLM

We build a small GPT-style language model and train it on Yelp reviews. The goal is to watch the model go from producing gibberish to coherent sentences as training progresses.

# 1 Setup

In [ ]:
#@title Import libraries { display-mode: "form" }
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from datasets import load_dataset
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, HTML
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass


---
# 2 Load Yelp Reviews

#### We grab 25,000 Yelp reviews and tokenize them into integer sequences. The training target is simple: given a sequence of tokens, predict the next one.

In [ ]:
dataset = load_dataset('yelp_review_full', split='train')

NUM_REVIEWS = 25000
texts = dataset['text'][:NUM_REVIEWS]

# Preview a sample of reviews
for i in range(50):
    stars = dataset['label'][i] + 1
    print(f"  [{stars}★] {texts[i][:100].replace(chr(10), ' ')}...")

In [ ]:
#@title Tokenize reviews { display-mode: "form" }
VOCAB_SIZE = 10000
SEQ_LEN = 64   # context window

# Convert raw text → integer token IDs
vectorizer = layers.TextVectorization(max_tokens=VOCAB_SIZE, output_sequence_length=SEQ_LEN + 1)
vectorizer.adapt(texts)
vocab = vectorizer.get_vocabulary()

print(f"Vocabulary: {len(vocab):,} tokens")
print(f"Sample:     {vocab[:15]}")
print(f"\n'the food was great' → {vectorizer(['the food was great']).numpy()[0][:5]}")

In [ ]:
#@title Build training sequences (input → next token) { display-mode: "form" }
all_tokens = vectorizer(np.array(texts)).numpy()
all_tokens = all_tokens[np.sum(all_tokens > 0, axis=1) > 20]  # drop very short reviews

# Input = tokens[:-1], Target = tokens shifted by one position
# This is how every LLM is trained: predict the next token
x_train = all_tokens[:, :-1]
y_train = all_tokens[:, 1:]

print(f"Training sequences: {x_train.shape[0]:,}")

---
# 3 Build the Model

#### Same architecture as GPT, just much smaller. The key ingredient is **causal masking**: each token can only attend to previous tokens, so the model can't cheat by looking ahead.

In [ ]:
#@title Token + position embeddings { display-mode: "form" }
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(vocab_size, embed_dim)  # what word?
        self.pos_emb   = layers.Embedding(maxlen, embed_dim)      # where in the sentence?

    def call(self, x):
        return self.token_emb(x) + self.pos_emb(tf.range(tf.shape(x)[-1]))

In [ ]:
# Transformer block with causal ("can't look ahead") attention
class CausalTransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.att  = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim // num_heads)
        self.ffn  = keras.Sequential([layers.Dense(ff_dim, activation='relu'), layers.Dense(embed_dim)])
        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()

    def call(self, x, training=False):
        attn = self.att(x, x, use_causal_mask=True)   # self-attention, no peeking forward
        x = self.norm1(x + attn)                       # residual + normalize
        return self.norm2(x + self.ffn(x))             # feed-forward + residual + normalize

In [ ]:
# Assemble the model
inputs  = layers.Input(shape=(SEQ_LEN,))
x = TokenAndPositionEmbedding(SEQ_LEN, VOCAB_SIZE, 128)(inputs)
x = CausalTransformerBlock(128, num_heads=4, ff_dim=256)(x)    # block 1
x = CausalTransformerBlock(128, num_heads=4, ff_dim=256)(x)    # block 2
outputs = layers.Dense(VOCAB_SIZE, activation='softmax')(x)    # predict next token

model = keras.Model(inputs, outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print(f"Parameters: {model.count_params():,}  (GPT-2 = 124M, GPT-4 ≈ 1.8T)")

---
# 4 Generation Helpers

#### Two utilities we'll reuse after each training phase. `show_predictions` displays the model's top guesses for the next token. The interactive widget lets you type any prompt and watch the model generate word by word — exactly how ChatGPT works.

In [ ]:
#@title Define generation utilities + interactive widget { display-mode: "form" }
id_to_word = dict(enumerate(vocab))

PROMPTS = ["the food was", "i would definitely",
           "the service at this restaurant", "we waited for"]


def _prompt_tokens(prompt):
    tokens = vectorizer([prompt]).numpy()[0]
    nonzero = np.where(tokens > 0)[0]
    if len(nonzero) == 0:
        return []
    return list(tokens[:nonzero[-1] + 1])


def show_predictions(model):
    """Print the model's top-5 next-token predictions for each prompt."""
    for prompt in PROMPTS:
        out = _prompt_tokens(prompt)
        if not out:
            continue
        padded = np.zeros(SEQ_LEN, dtype='int32')
        context = out[-SEQ_LEN:]
        padded[:len(context)] = context
        pred_pos = len(context) - 1
        probs = model.predict(padded[np.newaxis, :], verbose=0)[0][pred_pos]
        top5 = np.argsort(probs)[-5:][::-1]
        preds = ', '.join(f"{id_to_word[t]} ({probs[t]:.3f})" for t in top5)
        print(f'  "{prompt}" → {preds}')


def generate_text(model, prompt, length=50, temperature=0.8):
    """Generate text from a prompt."""
    out = _prompt_tokens(prompt)
    if not out:
        return "(empty prompt — try typing something)"

    prompt_len = len(out)
    for _ in range(length):
        padded = np.zeros(SEQ_LEN, dtype='int32')
        context = out[-SEQ_LEN:]
        padded[:len(context)] = context
        pred_pos = len(context) - 1

        p = model.predict(padded[np.newaxis, :], verbose=0)[0][pred_pos]
        p = np.exp(np.log(p + 1e-10) / temperature)
        p /= p.sum()
        t = np.random.choice(len(p), p=p)
        if t == 0:
            break
        out.append(t)

    prompt_words = [id_to_word[t] for t in out[:prompt_len]]
    generated_words = [id_to_word[t] for t in out[prompt_len:]]
    return f"<b>{' '.join(prompt_words)}</b> {' '.join(generated_words)}"


def make_generator_widget(model, title="Try it yourself"):
    """Create an interactive text box + generate button."""
    prompt_box = widgets.Text(
        value='the food was',
        placeholder='Type a prompt...',
        description='Prompt:',
        layout=widgets.Layout(width='500px'),
        style={'description_width': '60px'}
    )
    button = widgets.Button(description='Generate', button_style='primary')
    output_area = widgets.Output(layout=widgets.Layout(min_height='40px', padding='10px'))

    def run_generation(_=None):
        with output_area:
            output_area.clear_output(wait=True)
            display(HTML('<span style="color:#777">Generating...</span>'))
        html = generate_text(model, prompt_box.value)
        with output_area:
            output_area.clear_output(wait=True)
            display(HTML(f'<span style="font-size:15px">{html}</span>'))

    button.on_click(run_generation)
    prompt_box.on_submit(run_generation)

    display(HTML(f'<h4>{title}</h4>'))
    display(widgets.HBox([prompt_box, button]))
    display(output_area)


---
# 5 Phase 1 — 1 Epoch

In [ ]:
h1 = model.fit(x_train, y_train, batch_size=128, epochs=1, validation_split=0.05)

all_loss = list(h1.history['loss'])
all_val  = list(h1.history['val_loss'])
all_acc  = list(h1.history['accuracy'])

In [ ]:
show_predictions(model)

In [ ]:
make_generator_widget(model, "Phase 1 — Generate after 1 epoch")

---
# 6 Phase 2 — 5 Epochs Total

In [ ]:
h2 = model.fit(x_train, y_train, batch_size=128, epochs=4, validation_split=0.05)

all_loss += h2.history['loss']
all_val  += h2.history['val_loss']
all_acc  += h2.history['accuracy']

In [ ]:
show_predictions(model)

In [ ]:
make_generator_widget(model, "Phase 2 — Generate after 5 epochs")

---
# 7 Phase 3 — 10 Epochs Total

In [ ]:
h3 = model.fit(x_train, y_train, batch_size=128, epochs=5, validation_split=0.05)

all_loss += h3.history['loss']
all_val  += h3.history['val_loss']
all_acc  += h3.history['accuracy']

In [ ]:
show_predictions(model)

In [ ]:
make_generator_widget(model, "Phase 3 — Generate after 10 epochs")

---
# 8 Training Progress

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
epochs = range(1, len(all_loss) + 1)

ax1.plot(epochs, all_loss, 'b-o', ms=3, label='Train')
ax1.plot(epochs, all_val, 'r-o', ms=3, label='Val')
ax1.axvline(1, color='gray', ls='--', alpha=.5)
ax1.axvline(5, color='gray', ls='--', alpha=.5)
ax1.set(xlabel='Epoch', ylabel='Loss', title='Loss (lower = better predictions)')
ax1.legend()

ax2.plot(epochs, all_acc, 'b-o', ms=3)
ax2.axvline(1, color='gray', ls='--', alpha=.5)
ax2.axvline(5, color='gray', ls='--', alpha=.5)
ax2.set(xlabel='Epoch', ylabel='Accuracy', title='Next-Token Prediction Accuracy')

plt.tight_layout()
plt.show()